# Find a user collection and Explore it

- author : Sylvie Dagoret-Campagne
- creation date : 2026-10-10 : stage 1
- last update : 2026-10-16 : stage 2

- `fichier bps_dp2cosmos.sh`

```bash
export VISITIDS=$(paste -sd, cosmos_small.txt)
export CCDIDS=$(paste -sd, ccdIds.txt)

release="w_2026_40"

echo "run BPS"
bps_submit.sh --interactive bps_dp2cosmos_parsl.yaml
```

- `fichier bps_dp2cosmos_parsl.yaml`
```
pipelineYaml: "${DRP_PIPE_DIR}/pipelines/LSSTCam/DRP.yaml#stage1-single-visit"

payload:
  runInit: true
  payloadName: dp2cosmos202610
  butlerConfig: main
  inCollection: LSSTCam/defaults
  output: u/${USER}/LSSTCam/runcosmos01_20261008_RunCosmosDemo/stage1
  dataQuery: "instrument='LSSTCam' AND skymap='lsst_cells_v2' AND detector IN (${CCDIDS}) AND exposure IN (${VISITIDS}) AND visit IN (${VISITIDS})"
```

- the quantum graph for DRP.yaml#stage1-single-visit: https://tigress-web.princeton.edu/~lkelvin/pipelines/2026/w_2026_40/drp_pipe/LSSTCam/DRP/pipeline_drp_pipe_LSSTCam_DRP_stage1-single-visit.pdf
- all quantum graph : https://tigress-web.princeton.edu/~lkelvin/pipelines/2026/w_2026_40/drp_pipe/LSSTCam/DRP/

## Import

In [ ]:
import warnings

warnings.filterwarnings(
    "error",
    category=FutureWarning,
    message=".*Butler.collections.*",
)

In [ ]:
from lsst.daf.butler import Butler

butler_url = "main"
butler = Butler(butler_url)

## Identify existing user collections

In [ ]:
from lsst.daf.butler import CollectionType

prefix = "u/dagoret/"

collections = list(butler.collections.query(f"{prefix}*"))

for collection in collections:
    print(collection)

print(f"\nNombre de collections : {len(collections)}")


## 

### Choose the collection

In [ ]:
collection_path = (
    #stage 1 collection
    "u/dagoret/LSSTCam/test02_20261008_RunCosmosDemo/"
    "stage1/20261008T151954Z"
    #stage 2 collection
    #"u/dagoret/LSSTCam/test02_20261008_RunCosmosDemo/stage2/20261016T135502Z"
)

exists = any(
    name == collection_path
    for name in butler.collections.query(collection_path)
)
print(f"exists = {exists} for {collection_path}")

if collection_path in butler.collections.query(collection_path):
    info = butler.collections.get_info(collection_path)
    print(f"Collection trouvée : {collection_path}")
    print(f"Type : {info.type}")
else:
    print(f"Collection inexistante : {collection_path}")

## Find datasets in my collection

In [ ]:
import collections

### Total number of datasets

In [ ]:
refs = butler.registry.queryDatasets(
    datasetType="*",
    collections=collection_path,
    findFirst=False,
)

refs = list(refs)

print(f"Total number of datasets referenced  : {len(refs)} in {collection_path}")

counts = collections.Counter(
    ref.datasetType.name for ref in refs
)

for name, count in sorted(counts.items()):
    print(f"{name:40s} {count:8d}")

### Examine a number of references

In [ ]:
for ref in refs[:20]:
    print(
        ref.datasetType.name,
        ref.dataId
    )

### Examine post_isr_image

In [ ]:
for ref in butler.registry.queryDatasets(
    datasetType="post_isr_image",
    collections=collection_path,
    findFirst=False,
):
    print(ref.dataId)

Dans ta collection, il y a plusieurs catalogues liés aux étoiles. Pour le pipeline Rubin que tu as exécuté, les trois types les plus intéressants sont les suivants.


- Dataset type	    :->  Contenu / rôle 

- single_visit_star	: ->  Catalogue consolidé des étoiles détectées sur plusieurs visites, produit final de l'association des étoiles sur une visite unique.

- single_visit_star_detector :->	Catalogue des étoiles associé aux détecteurs individuels (179 datasets dans ta collection).

- isolated_star	    :-> Échantillons d'étoiles isolées, utilisés notamment pour la calibration photométrique et astrométrique.


- isolated_star_association	:-> Résultats de l'association des étoiles isolées entre observations.

  
- single_visit_psf_star	    :-> Étoiles sélectionnées pour la modélisation de la PSF, une sélection spécifique et non un catalogue stellaire général.


Je te conseille de commencer par single_visit_star, puis de regarder single_visit_star_detector si tu souhaites étudier les étoiles détectées sur chaque CCD.


### 1. Lire le catalogue single_visit_star

In [ ]:
refs = list(
    butler.registry.queryDatasets(
        datasetType="single_visit_star",
        collections=collection_path,
    )
)

print("Nombre de catalogues :", len(refs))

for ref in refs:
    print(ref.dataId)

### 2. Charger le catalogue dans un DataFrame

In [ ]:
import pandas as pd

ref = refs[0]

catalog = butler.get(ref)

print(type(catalog))
print(len(catalog))
#print(catalog.schema)

In [ ]:
catalog

In [ ]:
df = catalog.to_pandas()

display(df.head())
print(df.columns.tolist())

### 3. Examiner les étoiles détectées par détecteur

Pour accéder aux 179 catalogues single_visit_star_detector :

In [ ]:
refs_detector = list(
    butler.registry.queryDatasets(
        datasetType="single_visit_star_detector",
        collections=collection_path,
    )
)

print("Nombre de catalogues :", len(refs_detector))

catalog_detector = butler.get(refs_detector[0])

print(type(catalog_detector))


In [ ]:
df_det = catalog.to_pandas()

display(df_det.head())
print(df_det.columns.tolist())

### 4 Image

#### 4.1. Sélectionner un détecteur commun aux deux datasets
Dans ton notebook, commence par afficher les dataId disponibles :

In [ ]:
import pandas as pd

refs_img = list(
    butler.registry.queryDatasets(
        datasetType="post_isr_image",
        collections=collection_path,
    )
)

refs_cat = list(
    butler.registry.queryDatasets(
        datasetType="single_visit_star_detector",
        collections=collection_path,
    )
)

print("Images :", len(refs_img))
print("Catalogues :", len(refs_cat))

print("\nPremières images :")
for ref in refs_img[:5]:
    print(ref.dataId)

print("\nPremiers catalogues :")
for ref in refs_cat[:5]:
    print(ref.dataId)

Repère un couple qui possède le même visit et le même detector.
#### 4.2. Charger l'image et le catalogue
Une fois le dataId commun identifié, sélectionne les deux références correspondantes. Par exemple :

In [ ]:
print("IMAGE")
print(refs_img[0].dataId)
print("Dimensions :", list(refs_img[0].dataId.dimensions.names))

print("\nCATALOGUE")
print(refs_cat[0].dataId)
print("Dimensions :", list(refs_cat[0].dataId.dimensions.names))

In [ ]:
detectors_img = {
    r.dataId["detector"]
    for r in refs_img
}

detectors_cat = {
    r.dataId["detector"]
    for r in refs_cat
}

common_detectors = sorted(detectors_img & detectors_cat)

print("Nombre de détecteurs communs :", len(common_detectors))
print("Détecteurs :", common_detectors)

In [ ]:
detector_id = 47  # À remplacer par le détecteur choisi

ref_img = next(
    r for r in refs_img
    if r.dataId["detector"] == detector_id
)

ref_cat = next(
    r for r in refs_cat
    if r.dataId["detector"] == detector_id
    and r.dataId["visit"] == ref_img.dataId["exposure"]
)

exposure = butler.get(ref_img)
catalog = butler.get(ref_cat)

print("Image :", ref_img.dataId)
print("Catalogue :", ref_cat.dataId)
print("Dimensions image :", exposure.image.array.shape)
print("Type catalogue :", type(catalog))

#### 4.3 Charger les deux données

In [ ]:
exposure = butler.get(ref_img)
catalog = butler.get(ref_cat)

print("Type image :", type(exposure))
print("Type catalogue :", type(catalog))
print("Dimensions image :", exposure.image.array.shape)

In [ ]:
print(catalog.schema if hasattr(catalog, "schema") else catalog.colnames)